In [1]:
from pymongo import MongoClient
import pandas as pd
from bson import ObjectId
import re
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline, TFBertForSequenceClassification, BertTokenizer, create_optimizer
import torch
import torch.nn.functional as F
from datetime import datetime

In [2]:
LLM = "ElKulako/cryptobert"
FILENAME = "cryptobert_btc_sentiment.csv"
TWEETS_ID = '67082fb60891532d63a3ed67'
TRANSFORMATIONS = ["REMOVE_URLS", "REMOVE_HEX", "REMOVE_EMOTICONS", "REMOVE_USERNAMES"]
HASHTAGS_TO_KEEP = ["btc"]
STARTING_DATE = "2021-05-01"
ENDING_DATE = "2023-07-01"
RANDOM_SAMPLE = True
DAILY_SAMPLE_SIZE = 50

In [3]:
def remove_unwanted_hashtags(text, keep):
    def replacer(match):
        hashtag = match.group(0)[1:]
        if hashtag.lower() in keep:
            return match.group(0)
        else:
            return ''
    cleaned =  re.sub(r'[@$#]\w+', replacer, text)
    return re.sub(r'\s{2,}', ' ', cleaned).strip()

def apply_transformations(transformations, text):
    if "REMOVE_USERNAMES" in transformations:
        remove_usernames_pattern = r'@\S+'
        text = re.sub(remove_usernames_pattern, "", text)
    if "REMOVE_URLS" in transformations:
        remove_urls_pattern = r'http\S+'
        text = re.sub(remove_urls_pattern, "", text)
    if "REMOVE_PUNCTUATION_MARKS" in transformations:
        remove_pun_pattern_1 = r"(?<!\d)\.(?!\d)|[^\w\s.']"
        remove_pun_pattern_2 = r"'"
        remove_pun_pattern_3 = r"\s\s+"
        sub1 = re.sub(remove_pun_pattern_1, " ", text)
        sub2 = re.sub(remove_pun_pattern_2, "", sub1)
        text = re.sub(remove_pun_pattern_3, " ", sub2)
    if "TEXT_TO_LOWER" in transformations:
        text = text.lower()
    if "REMOVE_SHORT_WORDS" in transformations:
        remove_short_pattern = r'\b\w{1,2}\b'
        text = re.sub(remove_short_pattern, '', text)
    if "REMOVE_ENDING_HASHTAGS" in transformations:
        pattern = r'([#$@\$]\w+)(?=(\s[#$@\$]\w+)*\s*$)'
        text = re.sub(pattern, '', text)
    if "REMOVE_HEX" in transformations:
        text = re.sub(r'\b0x[a-fA-F0-9]{6,}\b', '', text)
    if "REMOVE_EMOTICONS" in transformations:
        emoji_pattern = re.compile(
            "[" 
            u"\U0001F600-\U0001F64F"  # Emoticons
            u"\U0001F300-\U0001F5FF"  # Symbols & pictographs
            u"\U0001F680-\U0001F6FF"  # Transport & map
            u"\U0001F1E0-\U0001F1FF"  # Flags
            u"\U00002700-\U000027BF"  # Dingbats
            u"\U0001F900-\U0001F9FF"  # Supplemental Symbols & Pictographs (includes 🤝)
            u"\U00002600-\U000026FF"  # Misc symbols (e.g. ☀️☂️)
            u"\U00002B00-\U00002BFF"  # Arrows etc.
            u"\U0001FA70-\U0001FAFF"  # Symbols and Pictographs Extended-A
            "]+", flags=re.UNICODE
        )
        text = emoji_pattern.sub(r'', text)
    return re.sub(r"\s{2,}", " ", text).strip()

In [4]:
client = MongoClient('mongodb://localhost:27017/')  
print('Connected to MongoDB')
db = client['phd']
tweets = db['tweets']
daily_tweets = db['daily_tweets']

Connected to MongoDB


In [5]:
all_tweets = pd.DataFrame(tweets.find({
    "tag_id": ObjectId(TWEETS_ID),
    "created_at": {
        "$gte": datetime.strptime(STARTING_DATE, "%Y-%m-%d"),
        "$lte": datetime.strptime(ENDING_DATE, "%Y-%m-%d").replace(hour=23, minute=59, second=59, microsecond=999999)
    }
}))
all_tweets['datetime'] = pd.to_datetime(all_tweets['created_at'])
all_tweets = all_tweets.sort_values(by='datetime')
all_tweets['datetime'] = all_tweets['datetime'].dt.date
all_tweets["content"] = all_tweets["content"].apply(lambda x: apply_transformations(TRANSFORMATIONS, remove_unwanted_hashtags(x, HASHTAGS_TO_KEEP)))

In [6]:
if RANDOM_SAMPLE == True:  
    all_tweets = all_tweets.groupby('datetime', group_keys=False).sample(n=DAILY_SAMPLE_SIZE, random_state=42)

In [7]:
# all_tweets = all_tweets.iloc[[1000, 1001, 10000, 10001, 100000, 100001]]

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(LLM)
finbert = AutoModelForSequenceClassification.from_pretrained(LLM)

id2label = {0: "positive", 1: "negative", 2: "neutral"}
sentiments = []
for index, tweet in all_tweets.iterrows():
    inputs = tokenizer(tweet["content"], return_tensors="pt", truncation=True)
    with torch.no_grad():
        logits = finbert(**inputs).logits
        probs = F.softmax(logits, dim=1).squeeze().tolist()
    sentiment_scores = {id2label[i]: probs[i] for i in range(3)}
    sentiments.append({ "date": tweet["datetime"], **sentiment_scores})

sentiments = pd.DataFrame(sentiments)

C:\Users\micha\anaconda32\lib\site-packages\huggingface_hub\file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


In [ ]:
sentiments_aggregated = sentiments.groupby('date', as_index=False)[['positive', 'negative', 'neutral']].mean()

In [ ]:
pd.DataFrame(sentiments_aggregated).to_csv(FILENAME)

In [ ]:
pd.DataFrame(sentiments_aggregated).plot()